In [1]:
# 在测试集上验证三个 YOLO 分割模型的 best.pt
# 数据集: D:\Develop\dataset\crack-multi-seg\data.yaml
from ultralytics import YOLO
from pathlib import Path

DATA_YAML = r"D:\Develop\dataset\crack-multi-seg\data.yaml"
BASE_DIR = Path.cwd() / "runs"

# 三个模型的最佳权重
models = {
    "segment (标准训练)":       BASE_DIR / "segment" / "train-2" / "weights" / "best.pt",
    "segment_coslr (余弦退火)": BASE_DIR / "segment_coslr" / "train-2" / "weights" / "best.pt",
    "segment_weighted (加权)":  BASE_DIR / "segment_weighted" / "train-3" / "weights" / "best.pt",
}

# 确认权重文件是否存在
for name, w in models.items():
    print(f"{'✅' if w.exists() else '❌'} {name}: {w}")

✅ segment (标准训练): d:\HP\OneDrive\Desktop\学校\课程\专业课\大数据综合工程设计\crack-seg\算法\runs\segment\train-2\weights\best.pt
✅ segment_coslr (余弦退火): d:\HP\OneDrive\Desktop\学校\课程\专业课\大数据综合工程设计\crack-seg\算法\runs\segment_coslr\train-2\weights\best.pt
✅ segment_weighted (加权): d:\HP\OneDrive\Desktop\学校\课程\专业课\大数据综合工程设计\crack-seg\算法\runs\segment_weighted\train-3\weights\best.pt


In [2]:
# 注册 WeightedSegmentationModel，用于加载 segment_weighted 的 checkpoint
import sys
from ultralytics.nn.tasks import SegmentationModel

module = sys.modules["__main__"]
if not hasattr(module, "WeightedSegmentationModel"):

    class WeightedSegmentationModel(SegmentationModel):
        pass

    module.WeightedSegmentationModel = WeightedSegmentationModel

print("✅ 自定义类已注册")

✅ 自定义类已注册


In [ ]:
# 逐一验证三个模型
all_results = {}

for name, weight_path in models.items():

    print(f"\n{'='*60}")
    print(f"  模型: {name}")
    print(f"  权重: {weight_path}")
    print(f"{'='*60}")

    model = YOLO(str(weight_path))

    results = model.val(
        data=DATA_YAML,
        split="test",       # 测试集
        imgsz=640,
        batch=64,
        device="cuda",
        verbose=True,
        save_json=False,
        plots=True,
        project=str(BASE_DIR / "test_results"),
        name=weight_path.parent.parent.name,
        exist_ok=True,
    )

    all_results[name] = results

    print(f"\n>>> {name} 测试集结果:")
    print(f"    mAP50(B)    : {results.box.map50:.4f}")
    print(f"    mAP50-95(B) : {results.box.map:.4f}")
    print(f"    mAP50(M)    : {results.seg.map50:.4f}")
    print(f"    mAP50-95(M) : {results.seg.map:.4f}")
    print(f"    各类别 mAP50-95(M): {[f'{c:.4f}' for c in results.seg.maps]}")
    print(f"    (注: 整体 mAP50(M) = 各类别 mAP50 均值; 各类别 mAP50 见上方 val 输出表格)")


  模型: segment (标准训练)
  权重: d:\HP\OneDrive\Desktop\学校\课程\专业课\大数据综合工程设计\crack-seg\算法\runs\segment\train-2\weights\best.pt
Ultralytics 8.4.60  Python-3.12.13 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO26m-seg summary (fused): 149 layers, 23,512,094 parameters, 0 gradients, 121.2 GFLOPs
val: Fast image access  (ping: 0.30.0 ms, read: 1001.0584.8 MB/s, size: 3616.9 KB)
val: Scanning D:\Develop\dataset\crack-multi-seg\test\labels.cache... 1096 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1096/1096  0.0s
val: D:\Develop\dataset\crack-multi-seg\test\images\gyu_12527.JPG: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 12.3s/it 3:4114.7s
                   all       1096       5466      0.434      0.419      0.379      0.225      0.384      0.362      0.294      0.125
                 Crack        112        148      0.

In [6]:
# 汇总对比三个模型的评价指标
import pandas as pd

rows = []
for name, r in all_results.items():
    rows.append({
        "模型": name,
        "mAP50(B)": f"{r.box.map50:.4f}",
        "mAP50-95(B)": f"{r.box.map:.4f}",
        "mAP50(M)": f"{r.seg.map50:.4f}",
        "mAP50-95(M)": f"{r.seg.map:.4f}",
    })

df = pd.DataFrame(rows)
print("\n" + "="*70)
print("  三个模型测试集结果对比")
print("="*70)
print(df.to_string(index=False))
print("="*70)

# 各类别详细 mAP50-95(M)
print("\n各类别 mAP50-95(M) 对比:\n")
cls_names = ["Crack", "Breakage", "Comb", "Hole", "Reinforcement", "Seepage"]

cls_rows = []
for name, r in all_results.items():
    cls_rows.append([name] + [f"{v:.4f}" for v in r.seg.maps])

cls_df = pd.DataFrame(cls_rows, columns=["模型"] + cls_names)
print(cls_df.to_string(index=False))

print("\n(各类别 mAP50(M) 请见上方每个模型 val 输出表格中的 Mask mAP50 列)")
print(f"按各类别 mAP50(M) 排序: Crack > Hole > Seepage > Reinforcement > Breakage > Comb")


  三个模型测试集结果对比
                   模型 mAP50(B) mAP50-95(B) mAP50(M) mAP50-95(M)
       segment (标准训练)   0.3788      0.2250   0.2936      0.1248
 segment_coslr (余弦退火)   0.3890      0.2281   0.2976      0.1243
segment_weighted (加权)   0.3757      0.2234   0.3071      0.1289

各类别 mAP50-95(M) 对比:

                   模型  Crack Breakage   Comb   Hole Reinforcement Seepage
       segment (标准训练) 0.2156   0.0952 0.0800 0.1214        0.1046  0.1319
 segment_coslr (余弦退火) 0.2108   0.0923 0.0802 0.1363        0.1018  0.1245
segment_weighted (加权) 0.2135   0.0935 0.0858 0.1285        0.1074  0.1445

(各类别 mAP50(M) 请见上方每个模型 val 输出表格中的 Mask mAP50 列)
按各类别 mAP50(M) 排序: Crack > Hole > Seepage > Reinforcement > Breakage > Comb


In [5]:
# 计算 Box 的 mAP@25 (IoU=0.25) 指标
# 需要同时 patch DetectionValidator.iouv (匹配阈值) 和 Metric.iou_thres (指标计算)
# try-finally 保证恢复 + 闭包提取原始函数避免递归
import torch
import numpy as np
from ultralytics.utils.metrics import Metric
from ultralytics.models.yolo.detect.val import DetectionValidator
import pandas as pd

print("=" * 70)
print("  计算 Box mAP@25 (IoU 阈值 = 0.25)")
print("=" * 70)

# ---- 获取真正的原始函数 ----
# DetectionValidator
_dv_init = DetectionValidator.__dict__["__init__"]
if _dv_init.__name__ == "_dv_patched_init" and _dv_init.__closure__:
    _dv_original_init = _dv_init.__closure__[0].cell_contents
else:
    _dv_original_init = _dv_init

# Metric
_m_init = Metric.__dict__["__init__"]
if _m_init.__name__ == "_m_patched_init" and _m_init.__closure__:
    _m_original_init = _m_init.__closure__[0].cell_contents
else:
    _m_original_init = _m_init


# ---- 定义补丁函数 ----
def _dv_patched_init(self, dataloader=None, save_dir=None, args=None, _callbacks=None):
    """DetectionValidator 补丁: 将 IoU 匹配阈值设为 [0.25]"""
    _dv_original_init(self, dataloader, save_dir, args, _callbacks)
    self.iouv = torch.tensor([0.25])
    self.niou = 1


def _m_patched_init(self):
    """Metric 补丁: 将 IoU 阈值设为 [0.25]"""
    _m_original_init(self)
    self.iou_thres = np.array([0.25])


# ---- 执行验证 ----
map25_data = {}

for name, weight_path in models.items():
    print(f"\n>>> 模型: {name}")

    # 同时打两个补丁
    DetectionValidator.__init__ = _dv_patched_init
    Metric.__init__ = _m_patched_init
    try:
        model = YOLO(str(weight_path))
        results = model.val(
            data=DATA_YAML,
            split="test",
            imgsz=640,
            batch=64,
            device="cuda",
            verbose=False,
            save_json=False,
            plots=False,
            project=str(BASE_DIR / "test_results"),
            name=f"{weight_path.parent.parent.name}_map25",
            exist_ok=True,
        )
    finally:
        # 无论是否异常都恢复
        DetectionValidator.__init__ = _dv_original_init
        Metric.__init__ = _m_original_init

    # box.map50 即为 mAP@25 (因为只有一个阈值)
    map25 = results.box.map50
    ap25_per_class = list(results.box.maps)

    print(f"    Box mAP@25 : {map25:.4f}")
    print(f"    各类 Box AP@25 : {[f'{c:.4f}' for c in ap25_per_class]}")

    map25_data[name] = {"Box mAP@25": map25, "Box AP@25": ap25_per_class}

# ---- 验证已正确恢复 ----
assert DetectionValidator.__init__ is _dv_original_init, "DetectionValidator 恢复失败!"
assert Metric.__init__ is _m_original_init, "Metric 恢复失败!"
print("\n✅ 所有补丁已正确恢复")

# ---- 汇总对比 ----
print("\n" + "=" * 70)
print("  Box mAP@25 汇总对比")
print("=" * 70)

cls_names = ["Crack", "Breakage", "Comb", "Hole", "Reinforcement", "Seepage"]
rows = []
for name, data in map25_data.items():
    row = {"模型": name, "Box mAP@25": f"{data['Box mAP@25']:.4f}"}
    for i, cn in enumerate(cls_names):
        row[cn] = f"{data['Box AP@25'][i]:.4f}"
    rows.append(row)

map25_df = pd.DataFrame(rows)
print(map25_df.to_string(index=False))
print("=" * 70)

# ---- 三个指标并列对比 ----
print("\n" + "=" * 110)
print("  三个指标并列对比:  Box mAP@25  |  mAP50(B)  |  mAP50-95(B)")
print("=" * 110)

combined_rows = []
for name, r in all_results.items():
    m25 = map25_data[name]["Box mAP@25"]
    combined_rows.append({
        "模型": name,
        "Box mAP@25": f"{m25:.4f}",
        "mAP50(B)": f"{r.box.map50:.4f}",
        "mAP50-95(B)": f"{r.box.map:.4f}",
    })

combined_df = pd.DataFrame(combined_rows)
print(combined_df.to_string(index=False))
print("=" * 110)
print("\nBox mAP@25: 检测框在 IoU=0.25 时的平均精度, 接近部署时的合并 IoU 阈值 0.30。")

  计算 Box mAP@25 (IoU 阈值 = 0.25)

>>> 模型: segment (标准训练)
Ultralytics 8.4.60  Python-3.12.13 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO26m-seg summary (fused): 149 layers, 23,512,094 parameters, 0 gradients, 121.2 GFLOPs
val: Fast image access  (ping: 0.30.1 ms, read: 832.5457.3 MB/s, size: 3973.7 KB)
val: Scanning D:\Develop\dataset\crack-multi-seg\test\labels.cache... 1096 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1096/1096  0.0s
val: D:\Develop\dataset\crack-multi-seg\test\images\gyu_12527.JPG: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 21.7s/it 6:3126.1s
                   all       1096       5466      0.487      0.465       0.45       0.45       0.47      0.447      0.414      0.414
Speed: 2.9ms preprocess, 341.2ms inference, 0.0ms loss, 1.0ms postprocess per image
    Box mAP@25 : 0.4497
    各类 Box

NameError: name 'all_results' is not defined